# Latent Altruism: Steering Cooperative Intent in LLMs
## NeurIPS 2026 — Complete Reproducibility Notebook

### What this notebook runs

**Script 1 — Core + Novel Experiments** (`kaggle_all_experiments.py`)
- Phase 1: Calibration (AllC / AllD hidden states, multi-layer)
- Phase 2: Baseline IPD games
- Phase 3: Prompt-Cooperative Baseline
- Phase 4: Control Vectors (Random + Orthogonal)
- Phase 5: Steered games — full α sweep (−2 to +1)
- Phase 6: Layer Ablation (extraction layer comparison)
- **Novel Exp A:** Layer 57 vs Last-Layer Injection *(Reviewer Q4)*
- **Novel Exp B:** Dynamic α Steering — Latent Tit-for-Tat
- **Novel Exp C:** Orthogonal Concept Erasure *(fix for Contextual Override)*
- **Novel Exp D:** Attention Head Importance at Layer 57 *(Veto Circuit)*

**Script 2 — Reviewer-Ready Deep Analytics** (`kaggle_reviewer_ready_experiments.py`)
- Module 1: Cross-Game Generalization (Stag Hunt, Chicken)
- Module 2: Adversarial Robustness with **full α + layer sweep** *(Reviewer Q5)*
- Module 3: Latent Space Visualization (PCA + t-SNE)
- Module 4: Layer Localization — FDI across ALL 64 layers
- Module 5: Method Comparison (SV vs CAA vs RepE)
- Module 6: **Proper PPL Benchmark** — 10 held-out sentences, mean ± std *(Reviewer Q2)*
- Module 7: Control Vector Ablation — Random + Orthogonal *(Reviewer concern)*

---
**Model:** Qwen/Qwen2.5-32B-Instruct (4-bit NF4, Kaggle H100 80 GB)

## Step 1: Clone Repository

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/trungkiet2005/steering_cooperative.git"
REPO_DIR = "/kaggle/working/steering_cooperative"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repository already present — pulling latest...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

os.chdir(REPO_DIR)
print("\nWorking directory:", os.getcwd())
!ls -lh experiments/

## Step 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
import bitsandbytes, torch, transformers
print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU detected")

## Step 3: Run Core + Novel Experiments

This script runs the main IPD steering experiment **plus** the four new contributions:
- Novel Exp A: Layer 57 Targeted Injection (forward-hook based)
- Novel Exp B: Dynamic α Steering — Latent Tit-for-Tat
- Novel Exp C: Orthogonal Concept Erasure
- Novel Exp D: Attention Head Importance at Layer 57

Expected runtime: **~3–5 hours** on H100

In [ ]:
!python experiments/kaggle_all_experiments.py 2>&1 | tee /kaggle/working/log_core.txt

## Step 3b (optional): Use pre-run core outputs from Kaggle dataset

If you **skip Step 3** and use the public [core-output dataset](https://www.kaggle.com/datasets/minh2duy/ouput-core-exp), run this cell to copy `steering_outputs` into working so Step 4 can read it.

In [ ]:
# import os
# import shutil

# # Path to steering_outputs inside the Kaggle dataset (add dataset: minh2duy/ouput-core-exp)
# CORE_OUTPUT_DATASET = "/kaggle/input/datasets/minh2duy/ouput-core-exp/steering_outputs"
# WORKING_OUTPUTS = "/kaggle/working/steering_outputs"

# if os.path.exists(CORE_OUTPUT_DATASET):
#     os.makedirs(WORKING_OUTPUTS, exist_ok=True)
#     for name in os.listdir(CORE_OUTPUT_DATASET):
#         src = os.path.join(CORE_OUTPUT_DATASET, name)
#         dst = os.path.join(WORKING_OUTPUTS, name)
#         if os.path.isfile(src):
#             shutil.copy2(src, dst)
#         elif os.path.isdir(src):
#             if os.path.exists(dst):
#                 shutil.rmtree(dst)
#             shutil.copytree(src, dst)
#     print(f"Copied core outputs from dataset to {WORKING_OUTPUTS}")
# else:
#     print("Dataset path not found. Add dataset 'minh2duy/ouput-core-exp' to the notebook, or run Step 3 first.")
#     print(f"Expected: {CORE_OUTPUT_DATASET}")

## Step 4: Run Reviewer-Ready Deep Analytics

This script runs all 7 deep-analysis modules:
- Full adversarial α sweep at both Last Layer and Layer 57
- Proper perplexity on 10 held-out sentences (mean ± std)
- Random + Orthogonal control vector ablation

Expected runtime: **~2–4 hours** on H100

In [ ]:
# Fix encoding (avoids SyntaxError: Non-UTF-8 on Kaggle after clone)
import os
_path = "experiments/kaggle_reviewer_ready_experiments.py"
if os.path.exists(_path):
    with open(_path, "rb") as _f:
        _raw = _f.read().replace(b"\x00", b"")
    for _bom in (b"\xff\xfe", b"\xfe\xff", b"\xef\xbb\xbf"):
        if _raw.startswith(_bom):
            _raw = _raw[len(_bom):]
            break
    if _raw[:1] in (b"\xff", b"\xfe"):
        try:
            _text = _raw.decode("utf-16-le")
        except Exception:
            try:
                _text = _raw.decode("utf-16-be")
            except Exception:
                _text = _raw.decode("utf-8", errors="replace")
    else:
        _text = _raw.decode("utf-8", errors="replace")
    if not _text.strip().startswith("# -*- coding"):
        _text = "# -*- coding: utf-8 -*-\n" + _text.lstrip("\ufeff")
    _text = _text.replace(chr(0), "")
    with open(_path, "w", encoding="utf-8") as _f:
        _f.write(_text)
    print("Encoding fix applied to", _path)

!python experiments/kaggle_reviewer_ready_experiments.py 2>&1 | tee /kaggle/working/log_deep.txt

## Step 5: Verify Output Files

In [ ]:
import os, json
import pandas as pd

out_core = "/kaggle/working/steering_outputs"
out_deep = "/kaggle/working/steering_outputs/deep_analysis"

print("=" * 55)
print("CORE OUTPUT FILES")
print("=" * 55)
for f in sorted(os.listdir(out_core)):
    fp = os.path.join(out_core, f)
    if os.path.isfile(fp):
        size_kb = os.path.getsize(fp) / 1024
        print(f"  {f:45s} {size_kb:7.1f} KB")

print("\n" + "=" * 55)
print("DEEP ANALYSIS OUTPUT FILES")
print("=" * 55)
if os.path.exists(out_deep):
    for f in sorted(os.listdir(out_deep)):
        fp = os.path.join(out_deep, f)
        if os.path.isfile(fp):
            size_kb = os.path.getsize(fp) / 1024
            print(f"  {f:45s} {size_kb:7.1f} KB")

## Step 6: Print Key Results

In [ ]:
# Core experiment summary
summary_path = f"{out_core}/experiment_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("CORE EXPERIMENT SUMMARY")
    print(json.dumps(summary, indent=2))

# Deep analysis summary
deep_path = f"{out_deep}/deep_analysis_summary.json"
if os.path.exists(deep_path):
    with open(deep_path) as f:
        deep_summary = json.load(f)
    print("\nDEEP ANALYSIS SUMMARY")
    print(json.dumps(deep_summary, indent=2))

## Step 7: Display Key Figures

### 7a. Core Results

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(path, title=""):
    if os.path.exists(path):
        print(f"\n{'─'*60}")
        print(f"  {title or os.path.basename(path)}")
        print(f"{'─'*60}")
        display(Image(filename=path, width=900))
    else:
        print(f"  [File not found: {path}]")

# Core plots
show_image(f"{out_core}/alpha_sweep.png",
           "Fig 1: Cooperation Rate vs α (full sweep, all opponents)")
show_image(f"{out_core}/condition_comparison.png",
           "Fig 2: Condition Comparison (Baseline vs Controls vs Steered)")
show_image(f"{out_core}/action_heatmap.png",
           "Fig 3: Action Heatmap (C/D grid, Baseline vs Best Steered)")

### 7b. Novel Experiments

In [ ]:
show_image(f"{out_core}/novel_a_layer_comparison.png",
           "Novel Exp A: Layer 57 vs Last Layer (Normal & Adversarial)")
show_image(f"{out_core}/novel_b_dynamic_steering.png",
           "Novel Exp B: Dynamic α Steering — Latent Tit-for-Tat")
show_image(f"{out_core}/novel_c_erasure_heatmap.png",
           "Novel Exp C: Orthogonal Concept Erasure vs Contextual Override")
show_image(f"{out_core}/novel_d_head_importance_layer57.png",
           "Novel Exp D: Attention Head Importance at Layer 57 (Veto Circuit)")

### 7c. Reviewer-Ready Deep Analytics

In [ ]:
show_image(f"{out_deep}/cross_game_generalization.png",
           "Module 1: Cross-Game Generalization (PD → Stag Hunt, Chicken)")
show_image(f"{out_deep}/adversarial_alpha_sweep.png",
           "Module 2: Adversarial Robustness — Full α Sweep (Last Layer vs Layer 57 vs OCE)")
show_image(f"{out_deep}/robustness_heatmap.png",
           "Module 2: Robustness Summary Heatmap")
show_image(f"{out_deep}/latent_space.png",
           "Module 3: Latent Space Geometry (PCA + t-SNE)")
show_image(f"{out_deep}/layer_localization.png",
           "Module 4: Layer Localization — FDI Across All 64 Layers")
show_image(f"{out_deep}/method_comparison.png",
           "Module 5: Method Comparison (SV vs CAA vs RepE)")
show_image(f"{out_deep}/ppl_benchmark.png",
           "Module 6: Perplexity Preservation (10 held-out sentences, mean ± std)")
show_image(f"{out_deep}/control_vector_ablation.png",
           "Module 7: Control Vector Ablation (Random vs Orthogonal)")

## Step 8: Print Aggregate Statistics Tables

In [ ]:
import pandas as pd

agg_path = f"{out_core}/aggregate_stats.csv"
if os.path.exists(agg_path):
    df_agg = pd.read_csv(agg_path)
    print("CORE AGGREGATE STATISTICS")
    print("=" * 80)
    # Show steered vs baseline
    key_conds = df_agg[df_agg["condition"].isin(["Baseline", "PromptCoop",
                                                   "RandomVec", "OrthogVec",
                                                   "Steered"])]
    with pd.option_context("display.float_format", "{:.3f}".format,
                           "display.max_rows", 100):
        display(key_conds[["condition", "alpha", "opponent", "n",
                            "coop_mean", "coop_std", "coop_ci95",
                            "payoff_mean"]].sort_values(
                    ["condition", "alpha", "opponent"]))

In [ ]:
# Novel Exp C: Orthogonal Concept Erasure results
nov_c_path = f"{out_core}/novel_c_erasure.csv"
if os.path.exists(nov_c_path):
    df_nc = pd.read_csv(nov_c_path)
    print("NOVEL EXP C: ORTHOGONAL CONCEPT ERASURE RESULTS")
    print("=" * 80)
    pivot = (df_nc.groupby(["adversarial", "condition"])["coop_rate"]
             .mean().unstack("condition"))
    with pd.option_context("display.float_format", "{:.3f}".format):
        display(pivot)

In [ ]:
# PPL benchmark table
ppl_path = f"{out_deep}/ppl_benchmark.csv"
if os.path.exists(ppl_path):
    df_ppl = pd.read_csv(ppl_path)
    print("MODULE 6: PERPLEXITY BENCHMARK (10 held-out sentences)")
    print("=" * 80)
    with pd.option_context("display.float_format", "{:.4f}".format):
        display(df_ppl[["label", "ppl_mean", "ppl_std",
                         "ratio_mean", "ratio_std"]])

## Step 9: Archive Results

In [ ]:
import shutil

archive = "/kaggle/working/neurips_results.tar.gz"
shutil.make_archive(
    base_name="/kaggle/working/neurips_results",
    format="gztar",
    root_dir="/kaggle/working",
    base_dir="steering_outputs",
)
size_mb = os.path.getsize(archive) / 1e6
print(f"Results archived: {archive}  ({size_mb:.1f} MB)")